In [1]:
import os
import sys

# Thêm path để import src
sys.path.append('/home/jovyan/work')

from src.spark_session import get_spark_session
from pyspark.sql.functions import col

# Khởi tạo Spark Session
spark = get_spark_session("verification-notebook")

In [2]:
# 1. Kiểm tra bảng đã tồn tại trong Catalog chưa
print("--- Danh sách bảng trong default db ---")
spark.sql("SHOW TABLES IN default").show()

--- Danh sách bảng trong default db ---
+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|olist_category_na...|      false|
|  default|     olist_customers|      false|
|  default|   olist_geolocation|      false|
|  default|   olist_order_items|      false|
|  default|olist_order_payments|      false|
|  default|        olist_orders|      false|
|  default|      olist_products|      false|
|  default|       olist_reviews|      false|
|  default|       olist_sellers|      false|
|  default|        silver_sales|      false|
+---------+--------------------+-----------+



In [3]:
# 2. Kiểm tra Schema của bảng Silver
print("--- Schema của bảng silver_sales ---")
spark.table("default.silver_sales").printSchema()

--- Schema của bảng silver_sales ---
root
 |-- product_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_

In [4]:
# 3. Kiểm tra dữ liệu mẫu (10 dòng)
print("--- Dữ liệu mẫu (10 dòng) ---")
spark.sql("SELECT order_id, customer_id, order_status, order_purchase_timestamp, order_delivered_customer_date FROM default.silver_sales LIMIT 10").show()

--- Dữ liệu mẫu (10 dòng) ---
+--------------------+--------------------+------------+------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_delivered_customer_date|
+--------------------+--------------------+------------+------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 03:56:33|          2017-10-10 14:25:13|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 13:41:37|          2018-08-07 08:27:45|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|   delivered|     2018-08-08 01:38:49|          2018-08-17 11:06:29|
|949d5b44dbf5de918...|f88197465ea7920ad...|   delivered|     2017-11-18 12:28:06|          2017-12-01 17:28:42|
|ad21c59c0840e6cb8...|8ab97904e6daea886...|   delivered|     2018-02-13 14:18:39|          2018-02-16 11:17:02|
|a4591c265e18cb1dc...|503740e9ca751ccdd...|   delivered|     2017-07-09 14

In [5]:
# 4. Kiểm tra việc xử lý Null và Filter Canceled
null_count = spark.table("default.silver_sales").filter(col("order_delivered_customer_date").isNull()).count()
canceled_count = spark.table("default.silver_sales").filter(col("order_status") == "canceled").count()
total_count = spark.table("default.silver_sales").count()

In [6]:
print(f"--- Thống kê ---")
print(f"Tổng số bản ghi: {total_count}")
print(f"Số bản ghi bị null ngày giao: {null_count} (Kỳ vọng: 0)")
print(f"Số bản ghi có trạng thái canceled: {canceled_count} (Kỳ vọng: 0)")

--- Thống kê ---
Tổng số bản ghi: 112719
Số bản ghi bị null ngày giao: 0 (Kỳ vọng: 0)
Số bản ghi có trạng thái canceled: 0 (Kỳ vọng: 0)


In [7]:
spark.stop()